# 🖼️ Digital Watermarking — LSB + Evaluasi Kompresi JPEG

**Tujuan:**
1. Sisipkan watermark biner ke foto wajah menggunakan metode **LSB** *(Least Significant Bit)*
2. Kompres foto dengan JPEG pada berbagai **Quality Factor (QF)**
3. Ekstrak watermark dari tiap hasil kompresi
4. Evaluasi dengan **PSNR**, **BER**, dan **NCC**
5. Temukan nilai QF minimum agar watermark masih bisa diekstrak

## 1. Install & Import Library

In [ ]:
# Library sudah tersedia di Colab, tidak perlu install tambahan
import numpy as np
from PIL import Image, ImageDraw, ImageFont
import io, os
import matplotlib.pyplot as plt
from IPython.display import display

print('✅ Library siap')

## 2. Upload Foto Wajah (Opsional)

Jika tidak upload, sistem akan membuat gambar sintetis secara otomatis.

In [ ]:
from google.colab import files

print('Upload foto wajah kamu (JPG/PNG), atau skip untuk pakai gambar sintetis:')
try:
    uploaded = files.upload()
    HOST_IMAGE_PATH = list(uploaded.keys())[0] if uploaded else None
    if HOST_IMAGE_PATH:
        print(f'✅ Foto berhasil diupload: {HOST_IMAGE_PATH}')
    else:
        print('⚠️ Tidak ada file diupload, akan pakai gambar sintetis.')
except Exception:
    HOST_IMAGE_PATH = None
    print('⚠️ Upload dilewati, akan pakai gambar sintetis.')

## 3. Konfigurasi

In [ ]:
# ── Konfigurasi ────────────────────────────────────────────
WATERMARK_TYPE   = 'binary'   # 'binary' (teks) atau 'random'
QUALITY_FACTORS  = [90, 80, 70, 60, 50, 40, 30, 20, 10]
WM_SIZE          = (64, 64)   # ukuran watermark (pixel)
BER_THRESHOLD    = 0.3        # BER > nilai ini → gagal ekstrak
NCC_THRESHOLD    = 0.5        # NCC < nilai ini → gagal ekstrak
OUTPUT_DIR       = 'output'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('✅ Konfigurasi OK')

## 4. Fungsi-Fungsi Utama

In [ ]:
# ── 4a. Generate Watermark ─────────────────────────────────

def generate_binary_watermark(size=(64,64), text='WM'):
    img = Image.new('L', size, color=0)
    draw = ImageDraw.Draw(img)
    try:
        font = ImageFont.load_default()
    except:
        font = None
    draw.text((4, 20), text, fill=255, font=font)
    wm = np.array(img)
    return (wm > 128).astype(np.uint8)

def generate_random_watermark(size=(64,64), seed=42):
    rng = np.random.default_rng(seed)
    return rng.integers(0, 2, size=size, dtype=np.uint8)

# ── 4b. Embed LSB ──────────────────────────────────────────

def embed_lsb(image_array, watermark):
    img = image_array.copy()
    wm_flat = watermark.flatten()
    h, w, _ = img.shape
    blue = img[:,:,2].flatten()
    blue[:len(wm_flat)] = (blue[:len(wm_flat)] & 0xFE) | wm_flat
    img[:,:,2] = blue.reshape(h, w)
    return img

# ── 4c. Kompres JPEG ───────────────────────────────────────

def compress_jpeg(image_array, quality):
    img_pil = Image.fromarray(image_array.astype(np.uint8))
    buf = io.BytesIO()
    img_pil.save(buf, format='JPEG', quality=quality)
    buf.seek(0)
    return np.array(Image.open(buf))

# ── 4d. Ekstrak LSB ────────────────────────────────────────

def extract_lsb(image_array, watermark_shape):
    n_bits = watermark_shape[0] * watermark_shape[1]
    blue = image_array[:,:,2].flatten()
    return (blue[:n_bits] & 1).reshape(watermark_shape).astype(np.uint8)

# ── 4e. Metrik ─────────────────────────────────────────────

def psnr(original, compressed):
    mse = np.mean((original.astype(np.float64) - compressed.astype(np.float64))**2)
    if mse == 0: return float('inf')
    return 10 * np.log10(255**2 / mse)

def ber(wm_orig, wm_ext):
    return np.mean(wm_orig.flatten() != wm_ext.flatten())

def ncc(wm_orig, wm_ext):
    a = wm_orig.astype(np.float64) - wm_orig.mean()
    b = wm_ext.astype(np.float64) - wm_ext.mean()
    denom = np.sqrt(np.sum(a**2) * np.sum(b**2))
    if denom == 0: return 0.0
    return float(np.sum(a*b) / denom)

print('✅ Fungsi siap')

## 5. Load Foto & Buat Watermark

In [ ]:
# Load foto host
if HOST_IMAGE_PATH and os.path.exists(HOST_IMAGE_PATH):
    host_img = np.array(Image.open(HOST_IMAGE_PATH).convert('RGB'))
    print(f'✅ Foto loaded: {host_img.shape}')
else:
    # Buat gambar sintetis 256×256
    h, w = 256, 256
    host_img = np.full((h, w, 3), [200, 170, 140], dtype=np.uint8)
    cx, cy = w//2, h//2
    for y in range(h):
        for x in range(w):
            if ((x-cx)/(w*0.35))**2 + ((y-cy)/(h*0.45))**2 < 1:
                host_img[y,x] = [220,185,155]
    host_img[cy-30:cy-20, cx-35:cx-15] = [50,30,20]
    host_img[cy-30:cy-20, cx+15:cx+35] = [50,30,20]
    host_img[cy+30:cy+38, cx-20:cx+20] = [160,80,80]
    print('✅ Gambar sintetis dibuat (256×256)')

# Buat watermark
if WATERMARK_TYPE == 'random':
    watermark = generate_random_watermark(WM_SIZE)
    print('✅ Watermark acak dibuat')
else:
    watermark = generate_binary_watermark(WM_SIZE, text='ITB')
    print('✅ Watermark biner (teks ITB) dibuat')

# Tampilkan
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(host_img); axes[0].set_title('Foto Host'); axes[0].axis('off')
axes[1].imshow(watermark, cmap='gray', vmin=0, vmax=1)
axes[1].set_title('Watermark Asli'); axes[1].axis('off')
plt.tight_layout(); plt.show()

## 6. Embed Watermark

In [ ]:
watermarked_img = embed_lsb(host_img, watermark)
diff = np.abs(watermarked_img.astype(int) - host_img.astype(int)).sum(axis=2)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(host_img); axes[0].set_title('Original'); axes[0].axis('off')
axes[1].imshow(watermarked_img); axes[1].set_title('Setelah Embed Watermark'); axes[1].axis('off')
axes[2].imshow(diff, cmap='hot'); axes[2].set_title('Perbedaan (diperbesar)'); axes[2].axis('off')
plt.suptitle(f'PSNR Embed: {psnr(host_img, watermarked_img):.2f} dB')
plt.tight_layout(); plt.show()
print('✅ Watermark berhasil disisipkan')

## 7. Evaluasi per Quality Factor

In [ ]:
results = []
print(f"{'QF':>5}  {'PSNR (dB)':>12}  {'BER':>8}  {'NCC':>8}  {'Status'}")
print('-' * 60)

for qf in QUALITY_FACTORS:
    compressed   = compress_jpeg(watermarked_img, quality=qf)
    extracted_wm = extract_lsb(compressed, watermark.shape)
    p = psnr(watermarked_img, compressed)
    b = ber(watermark, extracted_wm)
    n = ncc(watermark, extracted_wm)
    can_extract = (b <= BER_THRESHOLD) and (n >= NCC_THRESHOLD)
    results.append({'qf': qf, 'psnr': p, 'ber': b, 'ncc': n,
                    'can_extract': can_extract, 'compressed': compressed,
                    'extracted_wm': extracted_wm})
    status = '✓ Dapat diekstrak' if can_extract else '✗ GAGAL'
    p_str = f'{p:.2f}' if p != float('inf') else '∞'
    print(f"{qf:>5}  {p_str:>12}  {b:>8.4f}  {n:>8.4f}  {status}")

fail_qfs = [r['qf'] for r in results if not r['can_extract']]
if fail_qfs:
    threshold_qf = max(fail_qfs)
    print(f'\n🔴 Watermark TIDAK dapat diekstrak pada QF ≤ {threshold_qf}')
else:
    threshold_qf = None
    print('\n🟢 Watermark masih dapat diekstrak pada semua QF yang diuji.')

## 8. Visualisasi Metrik

In [ ]:
qfs   = [r['qf'] for r in results]
psnrs = [r['psnr'] if r['psnr'] != float('inf') else 100 for r in results]
bers  = [r['ber'] for r in results]
nccs  = [r['ncc'] for r in results]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Evaluasi Watermarking LSB vs Quality Factor JPEG', fontweight='bold')

axes[0].plot(qfs, psnrs, 'o-', color='steelblue', lw=2)
axes[0].set(title='PSNR (dB)', xlabel='QF', ylabel='PSNR (dB)')
axes[0].invert_xaxis(); axes[0].grid(True, alpha=0.4)

axes[1].plot(qfs, bers, 'o-', color='tomato', lw=2)
axes[1].axhline(BER_THRESHOLD, color='gray', ls='--', label=f'Threshold={BER_THRESHOLD}')
axes[1].set(title='BER (Bit Error Rate)', xlabel='QF', ylabel='BER')
axes[1].invert_xaxis(); axes[1].legend(); axes[1].grid(True, alpha=0.4)

axes[2].plot(qfs, nccs, 'o-', color='seagreen', lw=2)
axes[2].axhline(NCC_THRESHOLD, color='gray', ls='--', label=f'Threshold={NCC_THRESHOLD}')
axes[2].set(title='NCC', xlabel='QF', ylabel='NCC')
axes[2].invert_xaxis(); axes[2].legend(); axes[2].grid(True, alpha=0.4)

plt.tight_layout(); plt.show()

## 9. Grid Watermark Ter-Ekstrak per QF

In [ ]:
n = len(results)
cols = 5
rows = (n + cols - 1) // cols + 1

fig = plt.figure(figsize=(cols*3, rows*3))
fig.suptitle('Watermark Ter-Ekstrak per QF', fontweight='bold', fontsize=13)

ax = fig.add_subplot(rows, cols, 1)
ax.imshow(watermark, cmap='gray', vmin=0, vmax=1)
ax.set_title('Watermark\nAsli', fontsize=9); ax.axis('off')

ax2 = fig.add_subplot(rows, cols, 2)
ax2.imshow(watermarked_img)
ax2.set_title('Foto +\nWatermark', fontsize=9); ax2.axis('off')

for i, r in enumerate(results):
    ax = fig.add_subplot(rows, cols, cols + i + 1)
    ax.imshow(r['extracted_wm'], cmap='gray', vmin=0, vmax=1)
    status = '✓' if r['can_extract'] else '✗'
    color  = 'green' if r['can_extract'] else 'red'
    ax.set_title(f"QF={r['qf']} {status}\nBER={r['ber']:.3f}",
                 fontsize=8, color=color)
    ax.axis('off')

if threshold_qf:
    fig.text(0.5, 0.01, f'Watermark tidak dapat diekstrak pada QF ≤ {threshold_qf}',
             ha='center', fontsize=12, color='red', fontweight='bold')

plt.tight_layout(); plt.show()

## 10. Kesimpulan

In [ ]:
print('=' * 55)
print('KESIMPULAN')
print('=' * 55)
print(f'Metode embedding  : LSB (channel Biru)')
print(f'Ukuran watermark  : {WM_SIZE[0]} × {WM_SIZE[1]} bit = {WM_SIZE[0]*WM_SIZE[1]} bit total')
print(f'QF yang diuji     : {QUALITY_FACTORS}')
print()
for r in results:
    st = '✓ Dapat diekstrak' if r['can_extract'] else '✗ Tidak dapat diekstrak'
    psnr_str = f"{r['psnr']:.2f} dB" if r['psnr'] != float('inf') else '∞'
    print(f"  QF={r['qf']:>3}  PSNR={psnr_str:>10}  BER={r['ber']:.4f}  NCC={r['ncc']:.4f}  → {st}")

print()
if threshold_qf:
    print(f'🔴 Watermark TIDAK dapat diekstrak pada QF ≤ {threshold_qf}')
    print(f'   (LSB sangat rentan terhadap kompresi JPEG, karena')
    print(f'    kompresi lossy mengubah bit-bit LSB channel warna)')
else:
    print('🟢 Watermark masih dapat diekstrak pada semua QF yang diuji.')
print('=' * 55)